# Notebook da Tarefa 3 — Primeira Árvore de Decisão (CART)

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Disciplina:** Estrutura de Dados II / Redes de Computadores  
**Equipe:** Grupo 42  
**Integrantes:**  
- Wendell Rodrigues Macedo (Responsável / Scrum Master da Tarefa)  
- Vinicius Firmino Souto  
- Fernando Paiva Coelho  
- João Gustavo Sanches  
- Daniel Dias Paranhos  

---

### Objetivo e Contrato da Tarefa 3
- **Entrada (Congelada da Tarefa 2):**
  - Ficha de baseline congelada no Período A (`data/processed/baseline_por_fluxo.csv` — **não** recalculada com o Período B).
  - Partições temporais causais do Período B (`treino_periodo_b.csv` 50%, `validacao_periodo_b.csv` 20%, `teste_periodo_b.csv` 30% — **o teste permanece fechado até a Tarefa 5**).
- **Saídas desta Tarefa:**
  1. **Árvore de Decisão Oficial (CART)** ajustada (`fit`) exclusivamente no conjunto de treino, usando apenas as 8 métricas relativas ao baseline (`COLUNAS_MODELO`).
  2. **Regras da árvore em português** (copiadas das folhas reais da árvore) e descrição de profundidade, folhas e primeiras divisões.
  3. **Avaliação e leitura do erro na validação:** matriz 3×3 (`OK`, `RISCO`, `FALHA`), precisão, recall, F1 por classe, F1 macro e análise de casos concretos de erro.
  4. **Árvore de Contraste** (treinada com RTT absoluto em ms e rota) para comprovar que atalhos de distância condenam rotas longas saudáveis e mascaram falhas em rotas curtas — descartada formalmente da entrega final.

## 1. Configuração do Ambiente e Verificação do Contrato da Tarefa 2

In [1]:
import csv
import json
import math
from collections import Counter
from pathlib import Path

DIR_ATUAL = Path(".").resolve()
DIR_PROJETO = DIR_ATUAL.parent if DIR_ATUAL.name.lower() == "notebooks" else DIR_ATUAL
DIR_PROCESSED = DIR_PROJETO / "data" / "processed"
DIR_METADADOS = DIR_PROJETO / "metadados"

with open(DIR_PROCESSED / "baseline_por_fluxo.csv", encoding="utf-8") as f:
    baseline_rows = list(csv.DictReader(f))
with open(DIR_PROCESSED / "treino_periodo_b.csv", encoding="utf-8") as f:
    reader_tr = csv.DictReader(f)
    colunas_csv = list(reader_tr.fieldnames)
    train_rows = list(reader_tr)
with open(DIR_PROCESSED / "validacao_periodo_b.csv", encoding="utf-8") as f:
    val_rows = list(csv.DictReader(f))
with open(DIR_PROCESSED / "teste_periodo_b.csv", encoding="utf-8") as f:
    test_rows = list(csv.DictReader(f))

n_elegiveis = sum(1 for r in baseline_rows if r["status_baseline"] == "elegivel")
print("--- VERIFICAÇÃO DO CONTRATO DA TAREFA 2 ---")
print(f"Fichas de baseline (Período A congelado): {len(baseline_rows)} total | {n_elegiveis} elegíveis (>= 1.500 RTTs válidos)")
print(f"Bloco de Treino (50% mais antigo de B):    {len(train_rows):,}".replace(",", ".") + f" medições ({train_rows[0]['datetime_utc']} a {train_rows[-1]['datetime_utc']})")
print(f"Bloco de Validação (20% do meio de B):     {len(val_rows):,}".replace(",", ".") + f" medições ({val_rows[0]['datetime_utc']} a {val_rows[-1]['datetime_utc']})")
print(f"Bloco de Teste (30% mais recente de B):    {len(test_rows):,}".replace(",", ".") + " medições [MANTIDO FECHADO ATÉ A TAREFA 5]")
print("Ficha de baseline recalculada com Período B? False")

--- VERIFICAÇÃO DO CONTRATO DA TAREFA 2 ---
Fichas de baseline (Período A congelado): 67 total | 64 elegíveis (>= 1.500 RTTs válidos)
Bloco de Treino (50% mais antigo de B):    80.386 medições (2026-09-13T04:32:00Z a 2026-09-16T16:40:00Z)
Bloco de Validação (20% do meio de B):     32.148 medições (2026-09-16T16:20:00Z a 2026-09-18T02:12:00Z)
Bloco de Teste (30% mais recente de B):    48.270 medições [MANTIDO FECHADO ATÉ A TAREFA 5]
Ficha de baseline recalculada com Período B? False


## 2. Seleção Estrita de Colunas da Árvore Oficial (`COLUNAS_MODELO`)

- **Entram (`X`):** `z_robusto`, `aumento_pct`, `jitter_relativo`, `perda_pct`, `timeout_atual`, `n5_timeout`, `n5_aumento80`, `n5_risco`.
- **Alvo (`y`):** `classe` (`OK`, `RISCO`, `FALHA`).
- **Proibidas e excluídas de `X`:** `pais`, `dst_addr`, `prb_id`, `rota_id`, `fluxo_id`, `avg_rtt_ms`, `jitter_ms`, `mediana_baseline_ms`, `mad_baseline_ms`, `jitter_tipico_ms`, `timestamp`, `datetime_utc`, `ordem_regra`, `split`.

In [2]:
from src.arvore_inicial import (
    COLUNAS_MODELO,
    COLUNAS_ALVO,
    COLUNAS_PROIBIDAS_PRESENTES_NO_CSV,
    CLASSES,
    calcular_medianas_treino,
    construir_matriz_xy,
)

print("Todas as 23 colunas presentes no arquivo treino_periodo_b.csv:")
print(f"  {colunas_csv}\n")
print("Colunas usadas na matriz X da Árvore Oficial (8 features relativas):")
print(f"  {COLUNAS_MODELO}\n")
print(f"Coluna alvo (y): '{COLUNAS_ALVO}' -> Valores: {CLASSES}")
print(f"A coluna '{COLUNAS_ALVO}' está em COLUNAS_MODELO? {COLUNAS_ALVO in COLUNAS_MODELO}")
proibidas_presentes = any(c in COLUNAS_MODELO for c in COLUNAS_PROIBIDAS_PRESENTES_NO_CSV)
print(f"Alguma coluna de rota, IP ou RTT absoluto está em COLUNAS_MODELO? {proibidas_presentes}\n")

meds_treino = calcular_medianas_treino(train_rows, COLUNAS_MODELO)
print("Medianas aprendidas exclusivamente no Treino para imputação de NaN estruturais (timeouts/jitter sem 2 respostas):")
print(f"  { {k: round(v, 4) for k, v in meds_treino.items()} }")

X_tr, y_tr = construir_matriz_xy(train_rows, COLUNAS_MODELO, meds_treino)
X_va, y_va = construir_matriz_xy(val_rows, COLUNAS_MODELO, meds_treino)

Todas as 23 colunas presentes no arquivo treino_periodo_b.csv:
  ['timestamp', 'datetime_utc', 'fluxo_id', 'prb_id', 'dst_addr', 'sent', 'rcvd', 'avg_rtt_ms', 'jitter_ms', 'mediana_baseline_ms', 'mad_baseline_ms', 'jitter_tipico_ms', 'z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco', 'classe', 'ordem_regra', 'split']

Colunas usadas na matriz X da Árvore Oficial (8 features relativas):
  ['z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco']

Coluna alvo (y): 'classe' -> Valores: ['OK', 'RISCO', 'FALHA']
A coluna 'classe' está em COLUNAS_MODELO? False
Alguma coluna de rota, IP ou RTT absoluto está em COLUNAS_MODELO? False

Medianas aprendidas exclusivamente no Treino para imputação de NaN estruturais (timeouts/jitter sem 2 respostas):
  {'z_robusto': 0.4634, 'aumento_pct': 1.2828, 'jitter_relativo': 1.3039, 'perda_pct': 0.0, 'timeout_atual': 0.0, 'n5_time

## 3. Escolha de Hiperparâmetros na Validação e Ajuste (`fit`) da Árvore Oficial

Avaliamos a grade inicial de hiperparâmetros (`criterion` $\in \{\text{gini}, \text{entropy}\}$, `max_depth` $\in \{2, 3, 4\}$, `min_samples_leaf` $\in \{20, 50, 100\}$) ajustando (`fit`) **somente no treino** (`80.386` linhas) e medindo **somente na validação** (`32.148` linhas). Adotamos como **Primeira Árvore Oficial (Tarefa 3)** a configuração parcimoniosa:
- **Algoritmo:** Árvore de Decisão CART (`DecisionTreeClassifier`)
- **Critério:** `gini`
- **Profundidade máxima pedida (`max_depth`):** `3`
- **Mínimo de amostras por folha (`min_samples_leaf`):** `50`
- **Semente (`random_state`):** `42`

In [3]:
from src.arvore_inicial import ajustar_cart, estatisticas_arvore, avaliar_modelo

print("--- GRADE DE VALIDAÇÃO (FIT SÓ NO TREINO, AVALIAÇÃO SÓ NA VALIDAÇÃO) ---")
print("Critério | max_depth | min_leaf | Prof. Obtida | Folhas | Acurácia Val | F1 Macro Val | Recall FALHA | Recall RISCO")
print("-" * 116)
for crit in ["gini", "entropy"]:
    for md in [2, 3, 4]:
        msl = 50
        arv = ajustar_cart(X_tr, y_tr, list(range(len(X_tr))), 0, md, msl, COLUNAS_MODELO, crit)
        d_obt, n_folhas, _ = estatisticas_arvore(arv)
        av, _ = avaliar_modelo(arv, X_va, y_va)
        r_fa = av["por_classe"]["FALHA"]["recall"]
        r_ri = av["por_classe"]["RISCO"]["recall"]
        print(f"{crit:<8} | {md:9d} | {msl:8d} | {d_obt:12d} | {n_folhas:6d} | {av['acuracia']:12.4f} | {av['f1_macro']:12.4f} | {r_fa:12.4f} | {r_ri:12.4f}")

--- GRADE DE VALIDAÇÃO (FIT SÓ NO TREINO, AVALIAÇÃO SÓ NA VALIDAÇÃO) ---
Critério | max_depth | min_leaf | Prof. Obtida | Folhas | Acurácia Val | F1 Macro Val | Recall FALHA | Recall RISCO
--------------------------------------------------------------------------------------------------------------------
gini     |         2 |       50 |            2 |      4 |       0.9558 |       0.6373 |       0.9012 |       0.0000
gini     |         3 |       50 |            3 |      5 |       0.9803 |       0.6546 |       0.9617 |       0.0000
gini     |         4 |       50 |            4 |      6 |       0.9958 |       0.6655 |       1.0000 |       0.0000
entropy  |         2 |       50 |            2 |      3 |       0.9559 |       0.6374 |       0.9012 |       0.0000
entropy  |         3 |       50 |            3 |      4 |       0.9804 |       0.6547 |       0.9617 |       0.0000
entropy  |         4 |       50 |            4 |      5 |       0.9959 |       0.6655 |       1.0000 |       0.000

## 4. Estrutura da Árvore Oficial e Regras Lidas em Português

In [4]:
from src.arvore_inicial import exportar_texto_arvore, extrair_regras_portugues

arvore_oficial = ajustar_cart(X_tr, y_tr, list(range(len(X_tr))), 0, 3, 50, COLUNAS_MODELO, "gini")
prof_real, n_folhas, n_nos = estatisticas_arvore(arvore_oficial)

print("=== ÁRVORE DE DECISÃO OFICIAL (TAREFA 3) ===")
print("Critério: gini | max_depth pedido: 3 | min_samples_leaf: 50 | Semente: 42")
print(f"Profundidade real obtida: {prof_real} | Número de folhas: {n_folhas} | Total de nós: {n_nos}\n")
print("\n".join(exportar_texto_arvore(arvore_oficial)))

regras = extrair_regras_portugues(arvore_oficial)
print("\n=== REGRAS EXTRAÍDAS DA ÁRVORE (EM PORTUGUÊS) ===")
for idx, reg in enumerate(regras, 1):
    print(f"Regra {idx}: {reg['regra_pt']} (n_treino={reg['n_treino']}, gini={reg['impureza']:.4f})")

=== ÁRVORE DE DECISÃO OFICIAL (TAREFA 3) ===
Critério: gini | max_depth pedido: 3 | min_samples_leaf: 50 | Semente: 42
Profundidade real obtida: 3 | Número de folhas: 5 | Total de nós: 9

|--- z_robusto <= 3.4073 (n=80386, gini=0.4882)
|   |--- perda_pct <= 16.6650 (n=60797, gini=0.3527)
|   |   |--- n5_aumento80 <= 1.5000 (n=50514, gini=0.1301)
|   |   |   |--- classe: OK (n=48623, [OK=47000, RISCO=251, FALHA=1372], gini=0.0648)
|   |   |--- n5_aumento80 >  1.5000
|   |   |   |--- classe: FALHA (n=1891, [OK=0, RISCO=0, FALHA=1891], gini=0.0000)
|   |--- perda_pct >  16.6650
|   |   |--- classe: FALHA (n=10283, [OK=0, RISCO=0, FALHA=10283], gini=0.0000)
|--- z_robusto >  3.4073
|   |--- z_robusto <= 3.6010 (n=19589, gini=0.0006)
|   |   |--- classe: FALHA (n=50, [OK=5, RISCO=1, FALHA=44], gini=0.2152)
|   |--- z_robusto >  3.6010
|   |   |--- classe: FALHA (n=19539, [OK=0, RISCO=0, FALHA=19539], gini=0.0000)

=== REGRAS EXTRAÍDAS DA ÁRVORE (EM PORTUGUÊS) ===
Regra 1: Se z_robusto <= 3.

## 5. Avaliação e Leitura do Erro no Bloco de Validação

In [5]:
aval_val, preds_val = avaliar_modelo(arvore_oficial, X_va, y_va)
cm = aval_val["matriz_confusao"]

print("=== MATRIZ DE CONFUSÃO 3x3 NA VALIDAÇÃO (32.148 medições) ===")
print("Verdadeiro \\ Previsto |       OK |    RISCO |    FALHA |    Total")
print("-" * 65)
for c_t in CLASSES:
    tot_linha = sum(cm[c_t].values())
    print(f"{c_t:<21} | {cm[c_t]['OK']:>8,d} | {cm[c_t]['RISCO']:>8,d} | {cm[c_t]['FALHA']:>8,d} | {tot_linha:>8,d}".replace(",", "."))

print("\n=== MÉTRICAS POR CLASSE E GLOBAIS (VALIDAÇÃO) ===")
for c in CLASSES:
    m = aval_val["por_classe"][c]
    print(f"  Classe {c:<5} -> Precisão: {m['precisao']:.4f} | Recall: {m['recall']:.4f} | F1: {m['f1']:.4f} | Suporte: {m['suporte']:>6,d}".replace(",", "."))
print("  " + "-" * 73)
print(f"  F1 Macro (métrica principal): {aval_val['f1_macro']:.4f}")
print(f"  Balanced Accuracy:            {aval_val['balanced_accuracy']:.4f}")
print(f"  Acurácia Global:              {aval_val['acuracia']:.4f} ({aval_val['acuracia']*100:.2f}% — enganosa, pois mascara RISCO=0% e 500 FALHAs omitidas)")

=== MATRIZ DE CONFUSÃO 3x3 NA VALIDAÇÃO (32.148 medições) ===
Verdadeiro \ Previsto |       OK |    RISCO |    FALHA |    Total
-----------------------------------------------------------------
OK                    |   18.972 |        0 |        3 |   18.975
RISCO                 |      131 |        0 |        0 |      131
FALHA                 |      500 |        0 |   12.542 |   13.042

=== MÉTRICAS POR CLASSE E GLOBAIS (VALIDAÇÃO) ===
  Classe OK    -> Precisão: 0.9678 | Recall: 0.9998 | F1: 0.9836 | Suporte: 18.975
  Classe RISCO -> Precisão: 0.0000 | Recall: 0.0000 | F1: 0.0000 | Suporte:    131
  Classe FALHA -> Precisão: 0.9998 | Recall: 0.9617 | F1: 0.9803 | Suporte: 13.042
  -------------------------------------------------------------------------
  F1 Macro (métrica principal): 0.6546
  Balanced Accuracy:            0.6538
  Acurácia Global:              0.9803 (98.03% — enganosa, pois mascara RISCO=0% e 500 FALHAs omitidas)


## 6. Inspeção de Erros Concretos na Validação (Caminhos Curtos e Longos)

In [6]:
with open(DIR_METADADOS / "resumo_tarefa3.json", encoding="utf-8") as f:
    resumo_t3 = json.load(f)

aud = resumo_t3["auditoria_erros_validacao"]
print("--- CHECAGEM DE CAMINHO LONGO ESTÁVEL (BR->JP, BR->IN, BR->ZA) ---")
print(f"Total de medições OK de caminhos longos na validação: {aud['caminho_longo_estavel_ok_total_val']:,}".replace(",", "."))
print(f"Quantas caíram em FALHA na Árvore Oficial: {aud['caminho_longo_estavel_ok_caiu_em_falha_val']} (NENHUMA — o caso não apareceu na validação porque o modelo oficial usa z_robusto e não RTT absoluto)\n")

e1 = aud["exemplo_erro_1_falha_curto_em_ok"]
e2 = aud["exemplo_erro_2_pico_isolado_ok_em_falha"]
e3 = aud["exemplo_erro_3_risco_em_ok"]

print("--- ERROS CONCRETOS AUDITADOS NA VALIDAÇÃO ---")
print(f"1. [FALHA de caminho curto que caiu em OK] fluxo_id={e1['fluxo_id']} ({e1['rota']}, {e1['escopo']}) | ts={e1['datetime_utc']} ({e1['timestamp']})")
print(f"   RTT={e1['avg_rtt_ms']} ms (mediana={e1['mediana_baseline_ms']} ms) | z_robusto={e1['z_robusto']} | aumento_pct={e1['aumento_pct']}% | perda_pct={e1['perda_pct']}% | n5_timeout={e1['n5_timeout']} | n5_aumento80={e1['n5_aumento80']}")
print(f"   Classe verdadeira: {e1['classe_verdadeira']} (Ordem 2: n5_timeout >= 3) -> Classe da árvore: {e1['classe_arvore']}\n")

print(f"2. [OK de pico isolado em caminho curto que caiu em FALHA] fluxo_id={e2['fluxo_id']} ({e2['rota']}, {e2['escopo']}) | ts={e2['datetime_utc']} ({e2['timestamp']})")
print(f"   RTT={e2['avg_rtt_ms']} ms (mediana={e2['mediana_baseline_ms']} ms) | z_robusto={e2['z_robusto']} | aumento_pct={e2['aumento_pct']}% | perda_pct={e2['perda_pct']}% | n5_risco={e2['n5_risco']} (pico isolado)")
print(f"   Classe verdadeira: {e2['classe_verdadeira']} (Ordem 6: pico isolado sem persistência) -> Classe da árvore: {e2['classe_arvore']} (raiz cortou em z_robusto > 3.4073)\n")

print(f"3. [RISCO persistente que caiu em OK] fluxo_id={e3['fluxo_id']} ({e3['rota']}, {e3['escopo']}) | ts={e3['datetime_utc']} ({e3['timestamp']})")
print(f"   RTT={e3['avg_rtt_ms']} ms (mediana={e3['mediana_baseline_ms']} ms) | z_robusto={e3['z_robusto']} | aumento_pct={e3['aumento_pct']}% | perda_pct={e3['perda_pct']}% | n5_risco={e3['n5_risco']}")
print(f"   Classe verdadeira: {e3['classe_verdadeira']} (Ordem 5: 2 <= z_robusto < 3.5 e n5_risco >= 2) -> Classe da árvore: {e3['classe_arvore']} (max_depth=3 parou antes de abrir n5_risco)")

--- CHECAGEM DE CAMINHO LONGO ESTÁVEL (BR->JP, BR->IN, BR->ZA) ---
Total de medições OK de caminhos longos na validação: 4.530
Quantas caíram em FALHA na Árvore Oficial: 0 (NENHUMA — o caso não apareceu na validação porque o modelo oficial usa z_robusto e não RTT absoluto)

--- ERROS CONCRETOS AUDITADOS NA VALIDAÇÃO ---
1. [FALHA de caminho curto que caiu em OK] fluxo_id=101|194.109.133.10 (DE->DE, curto) | ts=2026-09-17T04:04:00Z (1789617840)
   RTT=15.32 ms (mediana=14.1000 ms) | z_robusto=1.4188 | aumento_pct=8.6525% | perda_pct=0.00% | n5_timeout=4 | n5_aumento80=0
   Classe verdadeira: FALHA (Ordem 2: n5_timeout >= 3) -> Classe da árvore: OK

2. [OK de pico isolado em caminho curto que caiu em FALHA] fluxo_id=102|194.109.133.10 (DE->DE, curto) | ts=2026-09-17T07:32:00Z (1789630320)
   RTT=17.14 ms (mediana=14.1300 ms) | z_robusto=3.4410 | aumento_pct=21.3022% | perda_pct=0.00% | n5_risco=1 (pico isolado)
   Classe verdadeira: OK (Ordem 6: pico isolado sem persistência) -> Classe d

## 7. Árvore de Contraste (Com RTT Absoluto e Rota — Fora da Entrega Final)

Para demonstrar por que `avg_rtt_ms`, `jitter_ms` e `rota_id` são proibidos, treinamos no mesmo conjunto de treino uma **Árvore de Contraste** que substitui o desvio normalizado pelo RTT absoluto (`avg_rtt_ms`) e pelo identificador da rota (`rota_id_num`).

In [7]:
c_rtt = resumo_t3["arvore_contraste"]["contraste_rtt_e_rota"]
print("=== ÁRVORE DE CONTRASTE A (avg_rtt_ms, perda_pct, rota_id_num) ===")
print(c_rtt["diagrama_texto"])
print("\n=== IMPACTO DO ATALHO DE DISTÂNCIA NA VALIDAÇÃO ===")
print(f"  F1 Macro da Árvore de Contraste: {c_rtt['avaliacao_validacao']['f1_macro']:.4f} (contra {aval_val['f1_macro']:.4f} da Árvore Oficial)")
print(f"  Medições OK de caminhos longos condenadas como FALHA: {c_rtt['longo_ok_caiu_em_falha_val']:,} na Contraste (contra 0 na Oficial)".replace(",", "."))
print(f"  Medições FALHA de caminhos curtos mascaradas como OK: {c_rtt['curto_falha_caiu_em_ok_val']:,} na Contraste (contra {aud['total_falha_curto_caiu_em_ok_val']} na Oficial)".replace(",", "."))
print("  DECLARAÇÃO: Esta árvore de contraste NÃO é o modelo do projeto e fica fora da entrega final.")

=== ÁRVORE DE CONTRASTE A (avg_rtt_ms, perda_pct, rota_id_num) ===
|--- perda_pct <= 16.6650 (n=80386, gini=0.4882)
|   |--- avg_rtt_ms <= 356.2850 (n=70047, gini=0.4438)
|   |   |--- avg_rtt_ms <= 328.8950 (n=66618, gini=0.4177)
|   |   |   |--- avg_rtt_ms <= 96.8150 (n=58681, gini=0.4410)
|   |   |   |   |--- classe: OK (n=46701, [OK=35171, RISCO=170, FALHA=11360], gini=0.3736)
|   |   |   |--- avg_rtt_ms >  96.8150
|   |   |   |   |--- classe: FALHA (n=11980, [OK=4446, RISCO=44, FALHA=7490], gini=0.4714)
|   |   |--- avg_rtt_ms >  328.8950
|   |   |   |--- avg_rtt_ms <= 352.4550 (n=7937, gini=0.1298)
|   |   |   |   |--- classe: OK (n=7832, [OK=7326, RISCO=23, FALHA=483], gini=0.1212)
|   |   |   |--- avg_rtt_ms >  352.4550
|   |   |   |   |--- classe: OK (n=105, [OK=60, RISCO=13, FALHA=32], gini=0.5653)
|   |--- avg_rtt_ms >  356.2850
|   |   |--- avg_rtt_ms <= 357.2350 (n=3429, gini=0.0023)
|   |   |   |--- classe: FALHA (n=89, [OK=2, RISCO=2, FALHA=85], gini=0.0869)
|   |   |--- 